# Day 1 â€” PySpark Practice Questions
## Topic: DataFrame Filtering Â· filter() Â· where() Â· isin() Â· between() Â· isNull()

---

> **Rules:**
> - These are the **exact same 3 problems as SQL Day 1** â€” same data, same business logic
> - Solve using the **PySpark DataFrame API** (not SparkSQL strings)
> - Each question has a setup cell that creates the DataFrame â€” run it first
> - Write your solution in the empty code cell below each question
> - Match the **exact expected output** (columns, values, order)

| # | Difficulty | Topic Focus |
|---|-----------|-------------|
| Q1 | ðŸŸ¢ Easy | `filter()` + `between()` + `isin()` |
| Q2 | ðŸŸ¡ Medium | `filter()` + `isNull()` + string methods + `when()` |
| Q3 | ðŸ”´ Hard | `groupBy()` + `agg()` + `filter()` (HAVING) + subquery equivalent |

---
## Setup â€” SparkSession (run once)

In [ ]:
import os, sys

# Must be set BEFORE any pyspark import â€” Spark reads these at import time
os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

print("Environment set. Run next cell to start Spark.")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, avg, count, sum as spark_sum,
    round as spark_round, year, to_date,
    lower, when, isnull, coalesce, concat_ws
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, FloatType, DoubleType, DateType
)

spark = SparkSession.builder \
    .appName("Day01_PracticeQuestions") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready.")

---
---

## Q1 â€” ðŸŸ¢ Easy
### Find Eligible Orders for a Discount Campaign

---

### Problem Statement

The marketing team is running a discount campaign. They want a list of orders that meet **all** of the following criteria:

1. The `order_amount` is between **500 and 5000** (inclusive)
2. The `status` is either `'delivered'` or `'confirmed'`
3. The order was placed in **2024**

Return `order_id`, `customer_name`, `order_amount`, `status`, and `order_date`.  
Sort by `order_amount` descending.

---

### DataFrame Schema

| Column | Type |
|--------|------|
| order_id | IntegerType |
| customer_name | StringType |
| order_amount | DoubleType |
| status | StringType |
| order_date | DateType |

In [ ]:
# Q1 Setup â€” run this cell first
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType
from pyspark.sql.functions import to_date

orders_data = [
    (1001, 'Rahul Sharma',  4500.00, 'delivered', '2024-01-15'),
    (1002, 'Priya Patel',    320.00, 'delivered', '2024-02-10'),  # amount < 500, excluded
    (1003, 'Amit Verma',    1200.00, 'confirmed', '2024-03-05'),
    (1004, 'Sneha Nair',    7800.00, 'delivered', '2024-03-20'),  # amount > 5000, excluded
    (1005, 'Rohan Gupta',   2900.00, 'cancelled', '2024-04-01'),  # status cancelled, excluded
    (1006, 'Meera Singh',    500.00, 'confirmed', '2024-04-18'),  # boundary: 500 included
    (1007, 'Vikram Joshi',  5000.00, 'delivered', '2024-05-22'),  # boundary: 5000 included
    (1008, 'Ananya Das',    3300.00, 'pending',   '2024-06-10'),  # status pending, excluded
    (1009, 'Sanjay Kumar',  1750.00, 'delivered', '2023-11-30'),  # year 2023, excluded
    (1010, 'Divya Reddy',   4100.00, 'confirmed', '2024-07-04'),
]

orders_schema = StructType([
    StructField("order_id",      IntegerType(), True),
    StructField("customer_name", StringType(),  True),
    StructField("order_amount",  DoubleType(),  True),
    StructField("status",        StringType(),  True),
    StructField("order_date",    StringType(),  True),
])

df_orders = spark.createDataFrame(orders_data, schema=orders_schema) \
                 .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

print("df_orders created â€” preview:")
df_orders.show(truncate=False)

### Expected Output

*(5 rows, sorted by order_amount DESC)*

```
+--------+-------------+------------+---------+----------+
|order_id|customer_name|order_amount|status   |order_date|
+--------+-------------+------------+---------+----------+
|1007    |Vikram Joshi |5000.0      |delivered|2024-05-22|
|1001    |Rahul Sharma |4500.0      |delivered|2024-01-15|
|1010    |Divya Reddy  |4100.0      |confirmed|2024-07-04|
|1003    |Amit Verma   |1200.0      |confirmed|2024-03-05|
|1006    |Meera Singh  |500.0       |confirmed|2024-04-18|
+--------+-------------+------------+---------+----------+
```

**PySpark API hints:**
- `col("order_amount").between(500, 5000)` for the range check
- `col("status").isin("delivered", "confirmed")` for status check
- `year(col("order_date")) == 2024` for the year check
- Combine with `&` â€” wrap each condition in `()`

In [ ]:
# Q1 â€” Write your solution here
# Use: filter() / where(), between(), isin(), year()



---
---

## Q2 â€” ðŸŸ¡ Medium
### Find Customers Who Need Profile Completion

---

### Problem Statement

The support team needs to contact customers whose profiles are incomplete.  
A profile is considered **incomplete** if **any** of the following are true:

1. `phone` is `null`
2. `email` does **not** contain `'@'` (malformed email)
3. `city` starts with `'unknown'` (case-insensitive)

Additionally, **only include** customers whose `account_status` is `'active'`.

Return `customer_id`, `full_name`, `email`, `phone`, `city`, and a computed column `issue` describing the **first matching** issue (priority order: phone â†’ email â†’ city).

Sort by `customer_id` ascending.

---

### DataFrame Schema

| Column | Type |
|--------|------|
| customer_id | IntegerType |
| full_name | StringType |
| email | StringType |
| phone | StringType (nullable) |
| city | StringType |
| account_status | StringType |

In [ ]:
# Q2 Setup â€” run this cell first
customers_data = [
    (1, 'Rahul Sharma',  'rahul@gmail.com',         '9876543210', 'Mumbai',          'active'),    # complete
    (2, 'Priya Patel',   'priya_at_yahoo_dot_com',   None,         'Delhi',           'active'),    # no phone + bad email
    (3, 'Amit Verma',    'amit@hotmail.com',         '9876543212', 'unknown city',    'active'),    # bad city
    (4, 'Sneha Nair',    'sneha@gmail.com',           None,         'Chennai',         'active'),    # no phone
    (5, 'Rohan Gupta',   'rohan@gmail.com',          '9876543214', 'Pune',            'inactive'),  # inactive
    (6, 'Meera Singh',   'meera_no_at_sign',         '9876543215', 'Hyderabad',       'active'),    # bad email
    (7, 'Vikram Joshi',  'vikram@gmail.com',         '9876543216', 'UNKNOWN REGION',  'active'),    # bad city
    (8, 'Ananya Das',    'ananya@gmail.com',         '9876543217', 'Kolkata',         'active'),    # complete
    (9, 'Sanjay Kumar',  'sanjay_broken_email',       None,         'Delhi',           'active'),    # no phone + bad email
    (10,'Divya Reddy',   'divya@gmail.com',          '9876543219', 'Bangalore',       'inactive'),  # inactive
]

customers_schema = StructType([
    StructField("customer_id",     IntegerType(), True),
    StructField("full_name",       StringType(),  True),
    StructField("email",           StringType(),  True),
    StructField("phone",           StringType(),  True),
    StructField("city",            StringType(),  True),
    StructField("account_status",  StringType(),  True),
])

df_customers = spark.createDataFrame(customers_data, schema=customers_schema)

print("df_customers created â€” preview:")
df_customers.show(truncate=False)

### Expected Output

*(6 rows â€” active customers with at least one profile issue, sorted by customer_id)*

```
+-----------+-------------+------------------------+----------+--------------+-------------+
|customer_id|full_name    |email                   |phone     |city          |issue        |
+-----------+-------------+------------------------+----------+--------------+-------------+
|2          |Priya Patel  |priya_at_yahoo_dot_com  |null      |Delhi         |missing phone|
|3          |Amit Verma   |amit@hotmail.com        |9876543212|unknown city  |unknown city |
|4          |Sneha Nair   |sneha@gmail.com         |null      |Chennai       |missing phone|
|6          |Meera Singh  |meera_no_at_sign        |9876543215|Hyderabad     |invalid email|
|7          |Vikram Joshi |vikram@gmail.com        |9876543216|UNKNOWN REGION|unknown city |
|9          |Sanjay Kumar |sanjay_broken_email     |null      |Delhi         |missing phone|
+-----------+-------------+------------------------+----------+--------------+-------------+
```

**PySpark API hints:**
- `col("phone").isNull()` for null check
- `~col("email").contains("@")` for malformed email
- `lower(col("city")).startswith("unknown")` for case-insensitive city check
- Build `issue` column using `when(...).when(...).otherwise(...)` chained expressions
- Filter the final DataFrame: keep rows where `issue` is not null

In [ ]:
# Q2 â€” Write your solution here
# Use: filter(), isNull(), contains(), lower(), startswith(), when()



---
---

## Q3 â€” ðŸ”´ Hard
### Flag Product Categories That Need Attention

---

### Problem Statement

The operations team wants to flag product categories that have **performance issues**.  
A category needs attention if it meets **at least one** of these conditions:

1. The **average sale amount** is **below the overall average** sale amount across all rows
2. The category has **more than 1 cancelled** order
3. The category has **fewer than 3 total orders**

For each flagged category, return:
- `category`
- `total_orders`
- `cancelled_orders`
- `avg_amount` (rounded to 2 decimal places)
- `flag` â€” a comma-separated string listing which conditions triggered
  - Use `'low avg'` for condition 1
  - Use `'high cancels'` for condition 2
  - Use `'few orders'` for condition 3

Sort by `total_orders` ascending.

---

### DataFrame Schema

| Column | Type |
|--------|------|
| sale_id | IntegerType |
| category | StringType |
| amount | DoubleType |
| status | StringType |

In [ ]:
# Q3 Setup â€” run this cell first
sales_data = [
    # Electronics: 5 orders, 1 cancelled, high avg
    (1,  'Electronics', 12000.0, 'completed'),
    (2,  'Electronics',  8500.0, 'completed'),
    (3,  'Electronics', 15000.0, 'completed'),
    (4,  'Electronics',  9200.0, 'cancelled'),
    (5,  'Electronics', 11000.0, 'completed'),
    # Clothing: 4 orders, 2 cancelled, low avg
    (6,  'Clothing',    1200.0, 'completed'),
    (7,  'Clothing',     800.0, 'cancelled'),
    (8,  'Clothing',    1500.0, 'cancelled'),
    (9,  'Clothing',     950.0, 'completed'),
    # Groceries: 2 orders, 0 cancelled, low avg
    (10, 'Groceries',   350.0, 'completed'),
    (11, 'Groceries',   420.0, 'completed'),
    # Furniture: 4 orders, 0 cancelled, high avg
    (12, 'Furniture',  18000.0, 'completed'),
    (13, 'Furniture',  22000.0, 'completed'),
    (14, 'Furniture',  16500.0, 'completed'),
    (15, 'Furniture',  19000.0, 'completed'),
    # Books: 2 orders, 2 cancelled, low avg
    (16, 'Books',       450.0, 'cancelled'),
    (17, 'Books',       380.0, 'cancelled'),
]

sales_schema = StructType([
    StructField("sale_id",  IntegerType(), True),
    StructField("category", StringType(),  True),
    StructField("amount",   DoubleType(),  True),
    StructField("status",   StringType(),  True),
])

df_sales = spark.createDataFrame(sales_data, schema=sales_schema)

print("df_sales created â€” preview by category:")
from pyspark.sql.functions import count as spark_count
df_sales.groupBy("category") \
        .agg(
            spark_count("*").alias("total_orders"),
            spark_count(when(col("status") == "cancelled", 1)).alias("cancelled_orders"),
            spark_round(avg("amount"), 2).alias("avg_amount")
        ) \
        .orderBy("category") \
        .show()

### Expected Output

*(3 flagged categories â€” Electronics and Furniture pass all conditions)*

```
+---------+------------+----------------+----------+---------------------------+
|category |total_orders|cancelled_orders|avg_amount|flag                       |
+---------+------------+----------------+----------+---------------------------+
|Books    |2           |2               |415.0     |few orders, high cancels, low avg|
|Groceries|2           |0               |385.0     |few orders, low avg        |
|Clothing |4           |2               |1112.5    |high cancels, low avg      |
+---------+------------+----------------+----------+---------------------------+
```

**Working it out:**
- Overall avg across all 17 rows = 157250 / 17 â‰ˆ 9250.0
- Electronics: avg=11140, 1 cancel, 5 orders â†’ no flags âœ“
- Furniture: avg=18875, 0 cancel, 4 orders â†’ no flags âœ“
- Clothing: avg=1112.5 (< 9250), 2 cancels (> 1), 4 orders â†’ `high cancels, low avg`
- Groceries: avg=385 (< 9250), 0 cancels, 2 orders (< 3) â†’ `few orders, low avg`
- Books: avg=415 (< 9250), 2 cancels (> 1), 2 orders (< 3) â†’ `few orders, high cancels, low avg`

**PySpark API hints:**
- Compute overall average first: `overall_avg = df_sales.agg(avg("amount")).collect()[0][0]`
- Use `count(when(col("status") == "cancelled", 1))` to count cancelled orders per category
- Build each flag condition with `when(..., lit("...")).otherwise(lit(None))` then use `concat_ws(", ", ...)` to join non-null flags
- Final `.filter()` to keep only rows where flag column is not null/empty

In [ ]:
# Q3 â€” Write your solution here
# Use: groupBy(), agg(), filter() (HAVING equivalent),
#      count(when(...)), avg(), when(), concat_ws(), lit()



In [ ]:
spark.stop()
print("SparkSession stopped.")